<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

# LangChain QA

All code comes from [LangChain docs](langchain.readthedocs.io).

In [1]:
# Core LangChain and OpenAI libraries
!pip install -qU langchain langchain-openai langchain-community langchain-community langchain-chroma langchain-text-splitters langchain-classic langgraph

# PDF handling and OCR libraries
!pip install -qU pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:

from langchain_classic.chains import RetrievalQA

# LLMs and Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

# Document loaders
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

# Text splitters
from langchain_text_splitters import CharacterTextSplitter

# Vector store integrations
from langchain_chroma import Chroma

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

print(llm.invoke("tell me a joke"))

content="Sure! Here's one for you:\n\nWhy don’t scientists trust atoms?  \nBecause they make up everything!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 11, 'total_tokens': 32, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_376a7ccef1', 'id': 'chatcmpl-Cw9lv5zeDPmfWrtydXnmkR6O4uc9L', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ba39e-59cd-7db3-9699-7a028c7b3423-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11, 'output_tokens': 21, 'total_tokens': 32, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


# load_qa_chain

Loads a chain that you can use to do QA over a set of documents, but it uses ALL of those documents.

chain_type="stuff" will not work because the number of tokens exceeds the limit. We can try other chain types like "map_reduce".

In [4]:
# Download PDF
!wget https://s3.amazonaws.com/weclouddata/datasets/tmp/example.pdf -O /content/example.pdf

--2026-05-16 21:42:55--  https://s3.amazonaws.com/weclouddata/datasets/tmp/example.pdf
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.253.238, 16.15.228.219, 52.217.234.16, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.253.238|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 374836 (366K) [application/pdf]
Saving to: ‘/content/example.pdf’

/content/example.pd 100%[===================>] 366.05K  1.46MB/s    in 0.2s    

2026-05-16 21:42:56 (1.46 MB/s) - ‘/content/example.pdf’ saved [374836/374836]



In [5]:
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_openai import OpenAI

# load document
loader = PyPDFLoader("/content/example.pdf")
documents = loader.load()

### For multiple documents
# loaders = [....]
# documents = []
# for loader in loaders:
#     documents.extend(loader.load())

chain = load_qa_chain(llm=OpenAI(), chain_type="map_reduce")
query = "what are the skills?"
chain.run(input_documents=documents, question=query)

/tmp/ipykernel_9087/1544935449.py:14: LangChainDeprecationWarning: The function `load_qa_chain` was deprecated in LangChain 0.2.13 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build new RAG flows with `create_agent` and a retrieval tool. See https://docs.langchain.com/oss/python/langchain/rag
  chain = load_qa_chain(llm=OpenAI(), chain_type="map_reduce")
/tmp/ipykernel_9087/1544935449.py:16: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain.run(input_documents=documents, question=query)


' The skills are excellent analytical and problem solving skills, ability to handle multiple projects, exceptionally well organized and resourceful, basic knowledge of MS Access, SQL, Visual Basic, C++, and proficiency in MS Office (Word, Excel, PowerPoint, Outlook) and QuickBooks.'

# RetrievalQA

RetrievalQA chain uses load_qa_chain under the hood. We retrieve the most relevant chunck of text and feed those to the language model.


#### Options:
- [embeddings](https://python.langchain.com/en/latest/reference/modules/embeddings.html)
- [TextSplitter](https://python.langchain.com/en/latest/modules/indexes/text_splitters.html)
- [VectorStore](https://python.langchain.com/en/latest/modules/indexes/vectorstores.html)
- [Retrievers](https://python.langchain.com/en/latest/modules/indexes/retrievers.html)
  - [search_type](https://python.langchain.com/en/latest/modules/indexes/vectorstores/examples/chroma.html#mmr): "similarity" or "mmr"
- [Chain Type](https://python.langchain.com/en/latest/modules/chains/index_examples/question_answering.html): "stuff", "map reduce", "refine", "map_rerank"


In [6]:
# load document
loader = PyPDFLoader("/content/example.pdf")
documents = loader.load()
# split the documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
# select which embeddings we want to use
embeddings = OpenAIEmbeddings()
# create the vectorestore to use as the index
db = Chroma.from_documents(texts, embeddings)
# expose this index in a retriever interface
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":2})
# create a chain to answer questions
qa = RetrievalQA.from_chain_type(
    llm=OpenAI(), chain_type="stuff", retriever=retriever, return_source_documents=True)
query = "what are the skills?"
result = qa({"query": query})

/tmp/ipykernel_9087/881746476.py:17: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa({"query": query})


In [7]:
retriever.invoke(query)

[Document(id='0ba57cf0-11ba-46cb-9437-6989889b4c60', metadata={'subject': 'career library handouts', 'creationdate': '2016-07-18T16:55:34-05:00', 'page': 0, 'creator': 'Microsoft® Word 2010', 'page_label': '1', 'total_pages': 1, 'source': '/content/example.pdf', 'title': 'Sample Resumes', 'author': 'career services/djg', 'keywords': 'Samples', 'moddate': '2016-07-26T08:54:50-05:00', 'producer': 'Microsoft® Word 2010'}, page_content='FUNCTIONAL (EXPERIENCED) \n \nIM A. SAMPLE I \n1234 North 55 Street \nBellevue, Nebraska 68005 \n(402) 292-2345 \nimasample1@xxx.com \n \nSUMMARY OF QUALIFICATIONS \nExceptionally well organized and resourceful Professional with more than six years experience and a \nsolid academic background in accounting and financial management; excellent analytical and problem \nsolving skills; able to handle multiple projects while producing high quality work in a fast-paced, \ndeadline-oriented environment. \n \nEDUCATION \nBachelor of Science, Bellevue University, Be

In [8]:
result

{'query': 'what are the skills?',
 'result': ' The skills mentioned in the context include accounting, financial management, analytical and problem solving, organization, resourcefulness, computer information systems, information systems analysis and problem solving, software conversion and reprogramming, research, bookkeeping, and proficiency in various computer programs. ',
 'source_documents': [Document(id='0ba57cf0-11ba-46cb-9437-6989889b4c60', metadata={'source': '/content/example.pdf', 'total_pages': 1, 'subject': 'career library handouts', 'creator': 'Microsoft® Word 2010', 'creationdate': '2016-07-18T16:55:34-05:00', 'page_label': '1', 'keywords': 'Samples', 'moddate': '2016-07-26T08:54:50-05:00', 'author': 'career services/djg', 'title': 'Sample Resumes', 'page': 0, 'producer': 'Microsoft® Word 2010'}, page_content='FUNCTIONAL (EXPERIENCED) \n \nIM A. SAMPLE I \n1234 North 55 Street \nBellevue, Nebraska 68005 \n(402) 292-2345 \nimasample1@xxx.com \n \nSUMMARY OF QUALIFICATIONS

# VectorstoreIndexCreator

VectorstoreIndexCreator is a wrapper for the above logic.

Source:
- https://python.langchain.com/en/latest/modules/chains/getting_started.html
- https://github.com/hwchase17/langchain/blob/master/langchain/indexes/vectorstore.py#L21-L74

In [9]:
from langchain_classic.indexes import VectorstoreIndexCreator

index = VectorstoreIndexCreator(
    # split the documents into chunks
    text_splitter=CharacterTextSplitter(chunk_size=1000, chunk_overlap=0),
    # select which embeddings we want to use
    embedding=OpenAIEmbeddings(),
    # use Chroma as the vectorestore to index and search embeddings
    vectorstore_cls=Chroma
).from_loaders([loader])
query = "what are the skills?"
index.query(llm=OpenAI(), question=query, chain_type="stuff")

' The skills mentioned in the context are organizational, analytical, problem-solving, multitasking, proficiency in various computer software, and knowledge of accounting and financial management. '

# ConversationalRetrievalChain

conversation memory + RetrievalQAChain

Allow for passing in chat history which can be used for follow up questions.

Source: https://python.langchain.com/en/latest/modules/chains/index_examples/chat_vector_db.html


In [10]:
from langchain_classic.chains import ConversationalRetrievalChain

In [11]:
# load document
loader = PyPDFLoader("/content/example.pdf")
documents = loader.load()
# split the documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
# select which embeddings we want to use
embeddings = OpenAIEmbeddings()
# create the vectorestore to use as the index
db = Chroma.from_documents(texts, embeddings)
# expose this index in a retriever interface
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":2})
# create a chain to answer questions
qa = ConversationalRetrievalChain.from_llm(OpenAI(), retriever)
chat_history = []
query = "what are the skills?"
result = qa({"question": query, "chat_history": chat_history})

In [12]:
chat_history = []
query = "what are the skills?"
result = qa({"question": query, "chat_history": chat_history})

In [13]:
result["answer"]

' The skills listed in the context include organizational, resourcefulness, analytical, problem-solving, multitasking, Microsoft Office proficiency, QuickBooks proficiency, basic knowledge of MS Access, SQL, Visual Basic, and C++.'

In [14]:
chat_history = [(query, result["answer"])]
query = "what are the skills related to technology?"
result = qa({"question": query, "chat_history": chat_history})

In [15]:
chat_history

[('what are the skills?',
  ' The skills listed in the context include organizational, resourcefulness, analytical, problem-solving, multitasking, Microsoft Office proficiency, QuickBooks proficiency, basic knowledge of MS Access, SQL, Visual Basic, and C++.')]

In [16]:
result['answer']

' Proficient in MS Office (Word, Excel, PowerPoint, Outlook), QuickBooks and Basic Knowledge of MS Access, SQL, Visual Basic, C++.'